# Basic Agentic Workflow

Helloooo everyone and welcome to an exciting lesson on Agentic AI! 🎉

Today, we're diving into **prompt chaining** agentic workflow pattern. What's that? It's just passing the output from one LLM to the next, step by step. Think of it like a relay race, but with prompts instead of batons.

We'll keep things super simple: manually run each cell, watch the magic happen, and see how chaining LLM calls lets us build more complex workflows.

Ready to see how agents can work together? Let's get started!

## As always, libraries first!

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

# check if API keys are set
if not OPENAI_API_KEY:
    raise ValueError("Missing OpenAI API key")
if not GEMINI_API_KEY:
    raise ValueError("Missing Gemini API key")
if not ANTHROPIC_API_KEY:
    raise ValueError("Missing Anthropic API key")

You can set your API Keys for each of the LLM providers using the following links:

- [OpenAI](https://platform.openai.com/api-keys)
- [Anthropic](https://console.anthropic.com/settings/keys)
- [Gemini](https://aistudio.google.com/app/apikey)

Once you have created the API Keys, you can store them on your `.env` file at the root of this repo

<div style="border-radius:16px;background:#2e3440;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
    <b style="color:#88c0d0;font-size:1.25em">Info:</b>
    <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6">
        <li>You can complete this entire notebook using just OpenAI models if you prefer!</li>
        <li>It's absolutely fine to skip Anthropic and Gemini for now — the workflow works perfectly with only OpenAI.</li>
        <li>Feel free to experiment with other providers later, but don't let missing API keys slow you down.</li>
    </ul>
    <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#88c0d0;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💡</div>
</div>

## The Workflow

```mermaid
graph LR
    A[Generate Tickets] --> B[Classify Priority] --> C[Respond to Tickets]
```

## Lets start with using OpenAI

In [2]:
# client
openai_client = OpenAI()

In [3]:
# messages list
message = "I want you to generate customer support ticket for a 3rd party re-seller. "
message += "The ticket should be a single sentence describing a common issue a customer might face with their product or service. "
message += "Please ensure the ticket is varied and covers different types of problems. "
message += "Do not give any subjects, only the body of the ticket."

messages = [{"role": "user", "content": message}]

In [4]:
# response

openai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

ticket = openai_response.choices[0].message.content
# print(f"### Generated Ticket:\n{ticket}")
display(Markdown(f"### Generated Ticket:\n{ticket}"))

### Generated Ticket:
I am unable to log into my account despite entering the correct credentials multiple times.

I love markdown. It is a lightweight method of rendering and formatting text that is super versatile without having to use heavy softwares like MS Word or Google Docs.

You can learn more about markdown sytanx [here](https://www.markdownguide.org/basic-syntax/)

A really informative YouTube video talking about the [Unreasonable Effectiveness of Plain Text](https://www.youtube.com/watch?v=WgV6M1LyfNY)

## Lets pass these on to an Anthropic model and ask it to classify the priority level of each ticket

In [5]:
# anthropic client, pass in base_url
anthropic_client = OpenAI(api_key=ANTHROPIC_API_KEY, base_url="https://api.anthropic.com/v1")

In [6]:
# messages list
message = "I want you to classify the priority of the following customer support ticket. "
message += "The ticket is as follows: "+ ticket + " "
message += "Please classify the priority as either 'Low', 'Medium', or 'High'. "
message += "Respond with only the priority level."

messages = [{"role": "user", "content": message}]

In [8]:
# response

openai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

priority = openai_response.choices[0].message.content
# print(f"### Generated Ticket:\n{ticket}")
display(Markdown(f"### Generated priority:\n{priority}"))

### Generated priority:
High

## Now Gemini should determine the appropriate response

In [9]:
# gemini client
gemini_client = OpenAI(api_key=GEMINI_API_KEY, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [10]:
# messages list

message = "You are to determine an appropriate response to the following customer support ticket. "
message += "The ticket is as follows: "+ ticket + " "
message += "The priority level of this ticket is: " + priority + " "
message += "Please provide a response that addresses the customer's issue in a short and concise manner. "
    
messages = [{"role": "user", "content": message}]

In [11]:
# response

gemini_response = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages
)

response = gemini_response.choices[0].message.content
display(Markdown(f"### Generated Response:\n{response}"))

### Generated Response:
Hello,

We understand you're unable to log into your account despite entering the correct credentials, and we're treating this with high priority.

To help us investigate and resolve this immediately, please reply with your registered email address or username.

We will look into this right away and get back to you within 30 minutes with an update or resolution.

<div style="border-radius:16px;background:#1e2a1e;margin:1em 0;padding:1em 1em 1em 3em;color:#eceff4;position:relative;box-shadow:0 6px 16px rgba(0,0,0,.4)">
    <b style="color:#a3be8c;font-size:1.25em">Your Challenge:</b>
    <ul style="margin:.6em 0 0;padding-left:1.2em;line-height:1.6"></ul>
        <li>Recreate the customer support ticket workflow using an <b>evaluator-optimizer agentic workflow pattern</b> instead of prompt chaining.</li>
        <li>Your evaluator agent should assess the quality and completeness of each ticket and suggest improvements.</li>
        <li>Your optimizer agent should revise the tickets based on evaluator feedback, aiming for clarity and actionable details.</li>
        <li>Try to implement this using at least two LLM calls (one for evaluation, one for optimization) and display the before/after results.</li>
        <li>Share your work in the community-contributions folder by creating a folder with your name. Eg. shaheer-airaj.</li>
    </ul>
    <div style="position:absolute;top:-.8em;left:-.8em;width:2.4em;height:2.4em;border-radius:50%;background:#a3be8c;color:#2e3440;display:flex;align-items:center;justify-content:center;font-weight:700;font-size:1.2em">💪</div>
</div>

# Now we need to evaluate each step and get LLMs to improve the ticket and responce

## We will use gemini to evaluate OpenAI work and OpenAI to evaluate gemini work

In [22]:
# Evaluate ticket created by OpenAI using Gemini 
evaluation_message = f"""
You are evaluating the quality of a customer support ticket generated by another model.

Original task:
Generate a customer support ticket for a 3rd-party reseller.

Requirements:
I want you to generate customer support ticket for a 3rd party re-seller. 
The ticket should be a single sentence describing a common issue a customer might face with their product or service. 
Please ensure the ticket is varied and covers different types of problems. 
Do not give any subjects, only the body of the ticket.

Generated ticket:
{ticket}

Evaluate the ticket for:
1. Clarity
2. Completeness
3. Accuracy against the task requirements

Be concise and direct.
If improvements are needed, list them.
If the ticket is good, say so briefly.
"""


evaluation_messages = [{"role": "user", "content": evaluation_message}]

In [23]:
# response

gemini_response = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=evaluation_messages
)

response = gemini_response.choices[0].message.content
display(Markdown(f"### Generated Response:\n{response}"))

### Generated Response:
**1. Clarity:** Good. The issue is clearly stated, along with relevant context (correct credentials, multiple attempts).
**2. Completeness:** Good. For a single sentence, it provides sufficient detail to understand the core problem.
**3. Accuracy against the task requirements:** Good.
    *   It is a single sentence.
    *   It describes a common issue.
    *   It does not include a subject.
    *   The issue described is generic enough to apply to a product or service offered by a 3rd-party reseller.

The ticket is good.

In [29]:
# Optimize ticket created by OpenAI using Gemini 
optimization_message = f"""
You are the optimizer in an evaluator-optimizer workflow.

Original task:
Generate a customer support ticket for a 3rd-party reseller.

Requirements:
I want you to generate customer support ticket for a 3rd party re-seller. 
The ticket should be a single sentence describing a common issue a customer might face with their product or service. 
Please ensure the ticket is varied and covers different types of problems. 
Do not give any subjects, only the body of the ticket.

Generated ticket:
{ticket}
You need to base your optimization on the evaluation of the ticket comleted by another model.
Evaluation response:
{response}

Your job:
Assess whether the generated ticket needs improvement based on the evaluator feedback.

If improvement is needed, rewrite the ticket so it better satisfies the original task.
If no improvement is needed, return the original ticket unchanged.

Return only the final ticket text.
Do not explain your reasoning.
Do not include labels such as "Optimized ticket:".
"""

optimization_messages  = [{"role": "user", "content": optimization_message}]

In [30]:
# response

gemini_response = gemini_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=optimization_messages
)

updated_ticket = gemini_response.choices[0].message.content
display(Markdown(f"### Generated Response:\n{updated_ticket}"))

### Generated Response:
I am unable to log into my account despite entering the correct credentials multiple times.

## In my example the ticket did not change. Lets see if the responce to the ticket will change
here we will evaluate gemini work using OpenAI 

In [37]:
# Evaluate ticket created by OpenAI using Gemini 
response_evaluation_message = f"""
You are evaluating the quality of a customer support ticket responce generated by another model.

Original task:
You are to determine an appropriate response to the following customer support ticket.
The ticket is as follows: {ticket}
The priority level of this ticket is: {priority}
Please provide a response that addresses the customer's issue in a short and concise manner.

Generated response:
{response}


Evaluate the ticket response for:
1. Clarity
2. Completeness
3. Accuracy against the task requirements

Be concise and direct.
If improvements are needed, list them.
If the ticket response is good, say so briefly.
"""

response_evaluation_messages = [{"role": "user", "content": response_evaluation_message}]

In [38]:
# response

openai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=response_evaluation_messages
)

response_evaluation = openai_response.choices[0].message.content
# print(f"### Generated Ticket:\n{ticket}")
display(Markdown(f"### Generated priority:\n{response_evaluation}"))

### Generated priority:
The ticket response is good. 

1. **Clarity:** Good. The issue is clearly articulated.
2. **Completeness:** Good. Sufficient detail is provided within a concise format.
3. **Accuracy against the task requirements:** Good. It aligns well with the task requirements. 

No improvements are needed.

In [40]:
# Optimize ticket response created by Gemini using OpenAI  
response_optimization_message = f"""
You are the optimizer in an evaluator-optimizer workflow.

Original task:
You are to determine an appropriate response to the following customer support ticket.
The ticket is as follows: {ticket}
The priority level of this ticket is: {priority}
Please provide a response that addresses the customer's issue in a short and concise manner.

Generated response:
{response}
You need to base your optimization on the evaluation of the ticket comleted by another model.
Evaluation response:
{response_evaluation}

Your job:
Assess whether the generated ticket response needs improvement based on the evaluator feedback.

If improvement is needed, rewrite the ticket response so it better satisfies the original task.
If no improvement is needed, return the original ticket response unchanged.

Return only the final ticket response.
Do not explain your reasoning.
Do not include labels such as "Optimized ticket:".
"""

response_optimization_messages  = [{"role": "user", "content": response_optimization_message}]

In [41]:
# response

openai_response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=response_optimization_messages
)

response_optimization = openai_response.choices[0].message.content
# print(f"### Generated Ticket:\n{ticket}")
display(Markdown(f"### Optimized Response:\n{response_optimization}"))

### Optimized Response:
Thank you for reaching out. We're sorry to hear you're having trouble logging into your account. Please try resetting your password using the "Forgot Password" link on the login page. If the issue persists, let us know, and we'll assist you further.

This model decided to change the responce, even so the evaluator said that the original is good.
I personaly prefer original to the new one